# Modules & Packages

As programs grow, keeping everything in one file becomes unmanageable. **Modules** let you split code across files. **Packages** group related modules into directories. The **standard library** ships hundreds of battle-tested modules with Python itself. And **third-party packages** (via `pip`) give you access to virtually everything else.

## What You'll Learn
1. What is a module?
2. `import` — all the ways to import
3. How Python finds modules — `sys.path`
4. Writing your own module
5. Packages & `__init__.py`
6. `__name__ == "__main__"` — the entry-point guard
7. Module introspection — `dir()`, `help()`, `__all__`
8. Standard library tour — `os`, `sys`, `pathlib`, `math`, `random`, `datetime`, `json`, `re`, `collections`, `itertools`
9. Virtual environments & `pip`
10. Relative vs absolute imports
11. Lazy imports & performance
12. Useful patterns — namespace packages, `importlib`

---

## 1. What Is a Module?

A **module** is simply a `.py` file. Any Python file you write is automatically a module. Modules let you:
- Organise related code in one place
- Reuse code across multiple programs
- Avoid name collisions by keeping names in separate **namespaces**

In [1]:
# Every module has a set of built-in attributes
import math

print(f"Name:     {math.__name__}")    # 'math'
print(f"File:     {math.__file__}")    # path to the compiled file
print(f"Package:  {math.__package__}") # '' (math is a top-level module)
print(f"Spec:     {math.__spec__}")    # module spec (loader, origin, etc.)
print(f"Doc:      {math.__doc__[:60]}...")

Name:     math
File:     /Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/lib-dynload/math.cpython-314-darwin.so
Package:  
Spec:     ModuleSpec(name='math', loader=<_frozen_importlib_external.ExtensionFileLoader object at 0x101735eb0>, origin='/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/lib-dynload/math.cpython-314-darwin.so')
Doc:      This module provides access to the mathematical functions
de...


In [2]:
# Modules are objects — you can inspect and pass them around
import os
import sys

print(type(os))         # <class 'module'>
print(isinstance(os, type(sys)))  # True — both are modules

# Modules are cached in sys.modules after first import
# Subsequent imports just return the cached object — no re-execution
print('math' in sys.modules)   # True — already imported above
print('csv'  in sys.modules)   # False — not yet imported

<class 'module'>
True
True
False


---

## 2. `import` — All the Ways to Import

In [3]:
# ── Form 1: import module ────────────────────────────────────────────────
# Imports the module; access its names via dot notation
import math

print(math.pi)          # 3.141592653589793
print(math.sqrt(16))    # 4.0
print(math.floor(3.9))  # 3

# ✅ Preferred for most cases — keeps namespace explicit

3.141592653589793
4.0
3


In [4]:
# ── Form 2: import module as alias ──────────────────────────────────────
# Shorter name for convenience, or to avoid name conflicts
import datetime as dt
import collections as col

today = dt.date.today()
print(today)

counter = col.Counter("mississippi")
print(counter)

# Convention: some libraries have universally accepted aliases
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt

2026-05-17
Counter({'i': 4, 's': 4, 'p': 2, 'm': 1})


In [5]:
# ── Form 3: from module import name ─────────────────────────────────────
# Import specific names directly into the current namespace
from math import pi, sqrt, factorial

print(pi)           # no 'math.' prefix needed
print(sqrt(25))
print(factorial(6))

3.141592653589793
5.0
720


In [6]:
# ── Form 4: from module import name as alias ─────────────────────────────
from datetime import datetime as DateTime, timedelta as Delta

now = DateTime.now()
next_week = now + Delta(days=7)
print(f"Now:       {now:%Y-%m-%d %H:%M}")
print(f"Next week: {next_week:%Y-%m-%d %H:%M}")

Now:       2026-05-17 12:28
Next week: 2026-05-24 12:28


In [7]:
# ── Form 5: from module import * ─────────────────────────────────────────
# Imports everything the module exports (names in __all__, or all public names)
# ⚠️ Generally discouraged — pollutes namespace and hides where names come from

from math import *
print(sin(pi / 2))   # 1.0 — works, but where did 'sin' come from?

# ✅ Exception: interactive sessions, small scripts, and some DSLs (e.g. turtle)

1.0


In [8]:
# ── Form 6: import inside a function ─────────────────────────────────────
# Defers the import until it's needed — useful for optional or slow dependencies

def render_template(text):
    import re   # only imported when this function is called
    return re.sub(r"\{(\w+)\}", "[REDACTED]", text)

print(render_template("Hello {name}, your token is {token}."))

Hello [REDACTED], your token is [REDACTED].


In [9]:
# ── Importing submodules ──────────────────────────────────────────────────
import os.path          # submodule
from os import path     # equivalent
from os.path import join, exists, dirname, basename

p = "/home/alice/projects/app.py"
print(os.path.exists(p))
print(dirname(p))
print(basename(p))
print(join("/home", "alice", "file.txt"))

False
/home/alice/projects
app.py
/home/alice/file.txt


---

## 3. How Python Finds Modules — `sys.path`

When you write `import foo`, Python searches a list of directories in order until it finds `foo.py` (or a `foo/` package). That list is `sys.path`.

In [10]:
import sys

print("sys.path entries:")
for i, p in enumerate(sys.path):
    print(f"  [{i}] {p or '(current directory)'}")

sys.path entries:
  [0] /Library/Frameworks/Python.framework/Versions/3.14/lib/python314.zip
  [1] /Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14
  [2] /Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/lib-dynload
  [3] (current directory)
  [4] /Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages
  [5] __editable__.finalproj-1.0.0.finder.__path_hook__


In [11]:
# sys.path is searched in order:
# 1. The directory of the script being run (or '' = current dir in interactive mode)
# 2. PYTHONPATH environment variable directories
# 3. Installation-dependent defaults (stdlib, site-packages)

# You can add to sys.path at runtime (generally avoid in production)
import sys, os

# sys.path.insert(0, '/my/custom/lib')   # highest priority
# sys.path.append('/my/other/lib')       # lowest priority

# Better: use a virtual environment or install the package properly

# Where does Python look for the standard library?
print(f"Prefix: {sys.prefix}")
print(f"Exec:   {sys.executable}")

Prefix: /Library/Frameworks/Python.framework/Versions/3.14
Exec:   /usr/local/bin/python3


In [12]:
# Finding where an installed module lives
import json, csv, re

for mod in [json, csv, re]:
    location = getattr(mod, '__file__', 'built-in')
    print(f"{mod.__name__:10} → {location}")

json       → /Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/json/__init__.py
csv        → /Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/csv.py
re         → /Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/re/__init__.py


---

## 4. Writing Your Own Module

Any `.py` file is a module. Let's create one from scratch and import it.

In [13]:
# Write a module file called 'mathutils.py'
module_code = '''
"""mathutils — simple mathematical utilities."""

__version__ = "1.0.0"
__author__  = "Tutorial"

# Module-level constant
TAU = 6.283185307179586   # 2 * pi

def clamp(value, lo, hi):
    """Clamp value to the range [lo, hi]."""
    return max(lo, min(value, hi))

def lerp(a, b, t):
    """Linear interpolation between a and b by factor t (0.0–1.0)."""
    return a + (b - a) * t

def is_prime(n):
    """Return True if n is a prime number."""
    if n < 2:
        return False
    for i in range(2, int(n ** 0.5) + 1):
        if n % i == 0:
            return False
    return True

def primes_up_to(n):
    """Return a list of all primes up to and including n."""
    return [x for x in range(2, n + 1) if is_prime(x)]

# Only runs when executed directly (not when imported)
if __name__ == "__main__":
    print(f"Primes up to 50: {primes_up_to(50)}")
'''

with open('mathutils.py', 'w') as f:
    f.write(module_code)

print("mathutils.py written.")

mathutils.py written.


In [14]:
# Now import and use it — just like any other module
import mathutils

print(mathutils.__doc__)
print(f"Version: {mathutils.__version__}")
print(f"TAU = {mathutils.TAU}")
print(f"clamp(15, 0, 10) = {mathutils.clamp(15, 0, 10)}")
print(f"lerp(0, 100, 0.3) = {mathutils.lerp(0, 100, 0.3)}")
print(f"is_prime(17) = {mathutils.is_prime(17)}")
print(f"primes up to 30: {mathutils.primes_up_to(30)}")

mathutils — simple mathematical utilities.
Version: 1.0.0
TAU = 6.283185307179586
clamp(15, 0, 10) = 10
lerp(0, 100, 0.3) = 30.0
is_prime(17) = True
primes up to 30: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]


In [15]:
# Import specific names from our module
from mathutils import clamp, lerp, primes_up_to

# Animate a value between 0 and 255 in 5 steps
steps = [lerp(0, 255, t/4) for t in range(5)]
clamped = [clamp(int(v), 0, 255) for v in steps]
print(f"Lerp 0→255: {clamped}")

print(f"Primes up to 50: {primes_up_to(50)}")

Lerp 0→255: [0, 63, 127, 191, 255]
Primes up to 50: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]


In [16]:
# Reloading a module after it's been changed
# (useful in development / interactive sessions)
import importlib
importlib.reload(mathutils)
print("Module reloaded.")

Module reloaded.


---

## 5. Packages & `__init__.py`

A **package** is a directory containing Python modules. A special file called `__init__.py` marks the directory as a package and runs when the package is first imported.

In [17]:
import os

# Create a simple package on disk
#
# geometry/
# ├── __init__.py
# ├── shapes.py
# └── transforms.py

os.makedirs('geometry', exist_ok=True)

# ── __init__.py ───────────────────────────────────────────────────────────
with open('geometry/__init__.py', 'w') as f:
    f.write('''
"""geometry — 2-D shape calculations."""

__version__ = "1.0.0"

# Re-export the most commonly used names at the top level
# so users can do: from geometry import Circle
from .shapes import Circle, Rectangle, Triangle
''')

# ── shapes.py ─────────────────────────────────────────────────────────────
with open('geometry/shapes.py', 'w') as f:
    f.write('''
"""Basic 2-D shapes."""
import math

class Circle:
    def __init__(self, radius: float):
        self.radius = radius
    def area(self):
        return math.pi * self.radius ** 2
    def perimeter(self):
        return 2 * math.pi * self.radius
    def __repr__(self):
        return f"Circle(radius={self.radius})"

class Rectangle:
    def __init__(self, width: float, height: float):
        self.width  = width
        self.height = height
    def area(self):
        return self.width * self.height
    def perimeter(self):
        return 2 * (self.width + self.height)
    def __repr__(self):
        return f"Rectangle(width={self.width}, height={self.height})"

class Triangle:
    def __init__(self, a: float, b: float, c: float):
        self.a, self.b, self.c = a, b, c
    def area(self):
        s = (self.a + self.b + self.c) / 2
        return math.sqrt(s * (s-self.a) * (s-self.b) * (s-self.c))
    def perimeter(self):
        return self.a + self.b + self.c
    def __repr__(self):
        return f"Triangle({self.a}, {self.b}, {self.c})"
''')

# ── transforms.py ─────────────────────────────────────────────────────────
with open('geometry/transforms.py', 'w') as f:
    f.write('''
"""Geometric transformations."""
import math

def translate(points, dx, dy):
    """Shift a list of (x, y) points by (dx, dy)."""
    return [(x + dx, y + dy) for x, y in points]

def scale(points, factor):
    """Scale a list of (x, y) points by factor."""
    return [(x * factor, y * factor) for x, y in points]

def rotate(points, angle_deg):
    """Rotate a list of (x, y) points around the origin."""
    rad = math.radians(angle_deg)
    cos_a, sin_a = math.cos(rad), math.sin(rad)
    return [
        (round(x*cos_a - y*sin_a, 6),
         round(x*sin_a + y*cos_a, 6))
        for x, y in points
    ]
''')

print("Package 'geometry' created.")

Package 'geometry' created.


In [18]:
# Import the package itself — __init__.py runs
import geometry
print(geometry.__version__)
print(geometry.__doc__)

# Names re-exported from __init__.py are available directly
from geometry import Circle, Rectangle, Triangle

c = Circle(5)
r = Rectangle(4, 6)
t = Triangle(3, 4, 5)

for shape in [c, r, t]:
    print(f"{shape!r:35}  area={shape.area():.2f}  perimeter={shape.perimeter():.2f}")

1.0.0
geometry — 2-D shape calculations.
Circle(radius=5)                     area=78.54  perimeter=31.42
Rectangle(width=4, height=6)         area=24.00  perimeter=20.00
Triangle(3, 4, 5)                    area=6.00  perimeter=12.00


In [19]:
# Import a submodule explicitly
from geometry import transforms

square = [(0,0), (1,0), (1,1), (0,1)]

print("Original:  ", square)
print("Translated:", transforms.translate(square, 3, 2))
print("Scaled:    ", transforms.scale(square, 2))
print("Rotated 45°:", transforms.rotate(square, 45))

Original:   [(0, 0), (1, 0), (1, 1), (0, 1)]
Translated: [(3, 2), (4, 2), (4, 3), (3, 3)]
Scaled:     [(0, 0), (2, 0), (2, 2), (0, 2)]
Rotated 45°: [(0.0, 0.0), (0.707107, 0.707107), (0.0, 1.414214), (-0.707107, 0.707107)]


In [20]:
# Package layout patterns
layout = """
Flat package (small project):
  mypackage/
  ├── __init__.py
  ├── core.py
  ├── utils.py
  └── config.py

Nested package (larger project):
  myapp/
  ├── __init__.py
  ├── api/
  │   ├── __init__.py
  │   ├── routes.py
  │   └── auth.py
  ├── db/
  │   ├── __init__.py
  │   ├── models.py
  │   └── migrations.py
  └── utils/
      ├── __init__.py
      ├── formatting.py
      └── validation.py
"""
print(layout)


Flat package (small project):
  mypackage/
  ├── __init__.py
  ├── core.py
  ├── utils.py
  └── config.py

Nested package (larger project):
  myapp/
  ├── __init__.py
  ├── api/
  │   ├── __init__.py
  │   ├── routes.py
  │   └── auth.py
  ├── db/
  │   ├── __init__.py
  │   ├── models.py
  │   └── migrations.py
  └── utils/
      ├── __init__.py
      ├── formatting.py
      └── validation.py



---

## 6. `__name__ == "__main__"` — The Entry-Point Guard

When Python runs a file, it sets `__name__` to `"__main__"` if it's the entry point, or to the module's name if it's being imported. This lets one file serve double duty — as both a runnable script and an importable module.

In [21]:
# In a Jupyter notebook, __name__ is always '__main__'
print(f"This notebook's __name__: {__name__}")

# But for our module:
import mathutils
print(f"mathutils.__name__:        {mathutils.__name__}")

This notebook's __name__: __main__
mathutils.__name__:        mathutils


In [22]:
# Write a script that doubles as a module
with open('greet_script.py', 'w') as f:
    f.write('''
"""Can be imported as a module OR run directly as a script."""

def make_greeting(name: str, formal: bool = False) -> str:
    if formal:
        return f"Good day, {name}."
    return f"Hey, {name}!"

def main():
    import sys
    name = sys.argv[1] if len(sys.argv) > 1 else "World"
    print(make_greeting(name))

if __name__ == "__main__":
    # This block only runs when executed as:
    #   python greet_script.py Alice
    # NOT when imported:
    #   import greet_script
    main()
''')

# Import it — main() is NOT called
import greet_script
print(greet_script.make_greeting("Alice"))
print(greet_script.make_greeting("Sir", formal=True))

Hey, Alice!
Good day, Sir.


In [23]:
# Run it as a script — main() IS called
import subprocess, sys
result = subprocess.run(
    [sys.executable, 'greet_script.py', 'Bob'],
    capture_output=True, text=True
)
print(f"Script output: {result.stdout.strip()}")

Script output: Hey, Bob!


---

## 7. Module Introspection — `dir()`, `help()`, `__all__`

In [24]:
import math

# dir() — list all names defined in a module
all_names = dir(math)
print(f"Total names: {len(all_names)}")

# Filter to just the public ones (no leading underscore)
public = [n for n in all_names if not n.startswith('_')]
print("Public names:", public)

Total names: 68
Public names: ['acos', 'acosh', 'asin', 'asinh', 'atan', 'atan2', 'atanh', 'cbrt', 'ceil', 'comb', 'copysign', 'cos', 'cosh', 'degrees', 'dist', 'e', 'erf', 'erfc', 'exp', 'exp2', 'expm1', 'fabs', 'factorial', 'floor', 'fma', 'fmod', 'frexp', 'fsum', 'gamma', 'gcd', 'hypot', 'inf', 'isclose', 'isfinite', 'isinf', 'isnan', 'isqrt', 'lcm', 'ldexp', 'lgamma', 'log', 'log10', 'log1p', 'log2', 'modf', 'nan', 'nextafter', 'perm', 'pi', 'pow', 'prod', 'radians', 'remainder', 'sin', 'sinh', 'sqrt', 'sumprod', 'tan', 'tanh', 'tau', 'trunc', 'ulp']


In [25]:
# help() — formatted documentation for a module or any of its members
help(math.gcd)    # just the one function
# help(math)      # full module docs — lots of output!

Help on built-in function gcd in module math:

gcd(*integers)
    Greatest Common Divisor.



In [26]:
# __all__ — explicitly declare the public API of your module
# Only names in __all__ are exported by 'from module import *'
# Best practice: always define __all__ in modules meant to be imported

with open('shapes2d.py', 'w') as f:
    f.write('''
"""2-D shape utilities with an explicit public API."""
import math

# Only these names will be exported by 'from shapes2d import *'
__all__ = ['circle_area', 'rect_area', 'PI']

PI = math.pi

def circle_area(r):
    return PI * r ** 2

def rect_area(w, h):
    return w * h

def _internal_helper():  # underscore = private by convention
    """Not part of the public API."""
    pass
''')

import shapes2d
print(f"__all__: {shapes2d.__all__}")
print(f"dir():   {[n for n in dir(shapes2d) if not n.startswith('__')]}")

__all__: ['circle_area', 'rect_area', 'PI']
dir():   ['PI', '_internal_helper', 'circle_area', 'math', 'rect_area']


In [27]:
# Introspecting any module with vars()
import json

# vars() returns the module's __dict__
module_vars = {k: type(v).__name__ for k, v in vars(json).items()
               if not k.startswith('_')}
for name, kind in list(module_vars.items())[:10]:
    print(f"  {name:20} ({kind})")

  scanner              (module)
  decoder              (module)
  JSONDecoder          (type)
  JSONDecodeError      (type)
  encoder              (module)
  JSONEncoder          (type)
  codecs               (module)
  dump                 (function)
  dumps                (function)
  detect_encoding      (function)


---

## 8. Standard Library Tour

Python's standard library contains over 200 modules. Here's a practical tour of the ones you'll use most often.

In [28]:
# ── os — operating system interface ──────────────────────────────────────
import os

print(f"CWD:        {os.getcwd()}")
print(f"Platform:   {os.name}")         # 'posix' / 'nt'
print(f"CPU count:  {os.cpu_count()}")
print(f"HOME:       {os.environ.get('HOME', 'N/A')}")
print(f"PATH items: {len(os.environ.get('PATH','').split(':'))}")

# File & directory operations
os.makedirs('demo_dir/subdir', exist_ok=True)
print(f"Exists: {os.path.exists('demo_dir')}")
print(f"Is dir: {os.path.isdir('demo_dir')}")
print(f"listdir: {os.listdir('demo_dir')}")

# Walk a directory tree
for root, dirs, files in os.walk('geometry'):
    level = root.count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")

CWD:        /Users/jacob/Code/Notes/Tutorials/Python-Tutorial/Lessons/01-Beginner/05-Modules
Platform:   posix
CPU count:  10
HOME:       /Users/jacob
PATH items: 44
Exists: True
Is dir: True
listdir: ['subdir']
geometry/
  transforms.py
  shapes.py
  __init__.py
  __pycache__/
    transforms.cpython-314.pyc
    __init__.cpython-314.pyc
    shapes.cpython-314.pyc


In [29]:
# ── pathlib — modern, object-oriented file paths ─────────────────────────
from pathlib import Path

p = Path('geometry')

print(f"Name:    {p.name}")
print(f"Suffix:  {p.suffix}")
print(f"Stem:    {p.stem}")
print(f"Parent:  {p.parent}")
print(f"Exists:  {p.exists()}")
print(f"Is dir:  {p.is_dir()}")

# Join paths with /
shapes_path = p / 'shapes.py'
print(f"shapes.py: {shapes_path}")
print(f"Size: {shapes_path.stat().st_size} bytes")

# Glob — find files matching a pattern
py_files = list(p.glob('*.py'))
print(f"Python files in geometry/: {[f.name for f in py_files]}")

# Read / write with Path
tmp = Path('tmp_example.txt')
tmp.write_text("Hello from pathlib!\n")
print(tmp.read_text())

Name:    geometry
Suffix:  
Stem:    geometry
Parent:  .
Exists:  True
Is dir:  True
shapes.py: geometry/shapes.py
Size: 1060 bytes
Python files in geometry/: ['transforms.py', 'shapes.py', '__init__.py']
Hello from pathlib!



In [30]:
# ── sys — interpreter internals ───────────────────────────────────────────
import sys

print(f"Python version: {sys.version}")
print(f"Version info:   {sys.version_info}")
print(f"Platform:       {sys.platform}")
print(f"Max int:        {sys.maxsize:,}")
print(f"Float info:     {sys.float_info.max:.3e} max")
print(f"Recursion limit:{sys.getrecursionlimit()}")
print(f"Arguments:      {sys.argv}")

# Memory size of an object
print(f"Size of []:     {sys.getsizeof([]):} bytes")
print(f"Size of {}:     {sys.getsizeof({}):} bytes")
print(f"Size of '':     {sys.getsizeof('')} bytes")

SyntaxError: f-string: valid expression required before '}' (2225520717.py, line 14)

In [ ]:
# ── math — mathematical functions ────────────────────────────────────────
import math

# Constants
print(f"pi  = {math.pi}")
print(f"e   = {math.e}")
print(f"tau = {math.tau}")
print(f"inf = {math.inf}")
print(f"nan = {math.nan}")

# Functions
print(f"sqrt(2)    = {math.sqrt(2):.6f}")
print(f"log(100)   = {math.log(100):.6f}")
print(f"log10(100) = {math.log10(100)}")
print(f"log2(8)    = {math.log2(8)}")
print(f"ceil(3.2)  = {math.ceil(3.2)}")
print(f"floor(3.9) = {math.floor(3.9)}")
print(f"trunc(3.9) = {math.trunc(3.9)}")
print(f"gcd(12,8)  = {math.gcd(12, 8)}")
print(f"lcm(4,6)   = {math.lcm(4, 6)}")
print(f"comb(5,2)  = {math.comb(5, 2)}")   # 5 choose 2
print(f"perm(5,2)  = {math.perm(5, 2)}")   # 5 permute 2
print(f"isclose:   {math.isclose(0.1+0.2, 0.3, rel_tol=1e-9)}")  # float comparison

In [ ]:
# ── random — random number generation ────────────────────────────────────
import random

random.seed(42)   # seed for reproducibility

# Numbers
print(f"random():          {random.random():.4f}")         # 0.0 – 1.0
print(f"uniform(1,10):     {random.uniform(1, 10):.4f}")  # float in range
print(f"randint(1,6):      {random.randint(1, 6)}")       # int, inclusive
print(f"randrange(0,100,5):{random.randrange(0,100,5)}")  # like range()
print(f"gauss(0,1):        {random.gauss(0, 1):.4f}")     # normal distribution

# Sequences
items = ['a', 'b', 'c', 'd', 'e']
print(f"choice:    {random.choice(items)}")
print(f"sample(3): {random.sample(items, 3)}")   # without replacement
print(f"choices(3):{random.choices(items, k=3)}") # with replacement

deck = list(range(1, 14))   # cards 1-13
random.shuffle(deck)
print(f"Shuffled:  {deck}")

In [ ]:
# ── datetime — dates, times, and durations ───────────────────────────────
from datetime import date, time, datetime, timedelta, timezone

# date
today = date.today()
print(f"Today:       {today}")
print(f"Year:        {today.year}")
print(f"Weekday:     {today.strftime('%A')}")  # e.g. 'Saturday'

# datetime
now = datetime.now()
print(f"Now:         {now}")
print(f"ISO format:  {now.isoformat()}")
print(f"Custom fmt:  {now.strftime('%B %d, %Y at %I:%M %p')}")

# timedelta arithmetic
birthday = date(1995, 8, 24)
age_days = (today - birthday).days
print(f"Days old:    {age_days:,}")

# Date arithmetic
in_100_days = today + timedelta(days=100)
print(f"In 100 days: {in_100_days}")

# Parsing a date string
parsed = datetime.strptime("2026-01-15 09:30", "%Y-%m-%d %H:%M")
print(f"Parsed:      {parsed}")

# Timezone-aware datetime
utc_now = datetime.now(timezone.utc)
print(f"UTC now:     {utc_now.isoformat()}")

In [ ]:
# ── json — JSON encoding and decoding ────────────────────────────────────
import json

data = {
    "name": "Alice",
    "age": 30,
    "scores": [92.5, 87, 95],
    "active": True,
    "address": None
}

# Python → JSON string
json_str = json.dumps(data, indent=2)
print("JSON string:")
print(json_str)

# JSON string → Python
loaded = json.loads(json_str)
print(f"\nLoaded name: {loaded['name']}")
print(f"Type: {type(loaded)}")

# Write JSON to a file
with open('data.json', 'w') as f:
    json.dump(data, f, indent=2)

# Read JSON from a file
with open('data.json') as f:
    from_file = json.load(f)
print(f"From file: {from_file['name']}, {from_file['scores']}")

In [ ]:
# ── re — regular expressions ─────────────────────────────────────────────
import re

text = "Contact us at hello@example.com or support@company.org for help."

# search — find first match anywhere in the string
match = re.search(r'[\w.+-]+@[\w-]+\.[\w.]+', text)
if match:
    print(f"First email: {match.group()}")

# findall — find all non-overlapping matches
emails = re.findall(r'[\w.+-]+@[\w-]+\.[\w.]+', text)
print(f"All emails:  {emails}")

# sub — replace matches
censored = re.sub(r'[\w.+-]+@[\w-]+\.[\w.]+', '[REDACTED]', text)
print(f"Censored:    {censored}")

# Groups — capture parts of the match
pattern = r'(\w+)@([\w.]+)'
for match in re.finditer(pattern, text):
    user, domain = match.groups()
    print(f"  user={user}, domain={domain}")

# Compile a pattern when using it many times
phone_re = re.compile(r'\b\d{3}[-.]\d{3}[-.]\d{4}\b')
phones = phone_re.findall("Call 555-123-4567 or 800.555.0100")
print(f"Phones: {phones}")

In [ ]:
# ── collections — specialised container types ────────────────────────────
from collections import Counter, defaultdict, OrderedDict, namedtuple, deque, ChainMap

# Counter — count hashable objects
words = "the quick brown fox jumps over the lazy dog the fox".split()
c = Counter(words)
print(f"Counter: {c}")
print(f"Most common 3: {c.most_common(3)}")
print(f"'the' appears {c['the']} times")
print()

# defaultdict — dict with a default value factory
groups = defaultdict(list)
for word in words:
    groups[len(word)].append(word)   # group by length; no KeyError!
for length, ws in sorted(groups.items()):
    print(f"  {length} letters: {ws}")
print()

In [ ]:
from collections import namedtuple, deque, ChainMap

# namedtuple — lightweight, immutable record
Point  = namedtuple('Point', ['x', 'y'])
Person = namedtuple('Person', ['name', 'age', 'city'])

p = Point(3, 4)
print(f"Point: {p}, x={p.x}, y={p.y}")
print(f"Distance from origin: {(p.x**2 + p.y**2)**0.5:.2f}")

alice = Person("Alice", 30, "NYC")
print(alice)
print(alice._asdict())   # convert to dict
print()

# deque — double-ended queue, O(1) append/pop on both ends
dq = deque([1, 2, 3], maxlen=5)
dq.append(4)       # right
dq.appendleft(0)   # left
print(f"deque: {dq}")
print(f"popleft: {dq.popleft()}, pop: {dq.pop()}")
print(f"After pops: {dq}")
dq.rotate(1)       # rotate right
print(f"Rotated: {dq}")
print()

# ChainMap — view multiple dicts as one
defaults  = {'color': 'blue', 'font': 'Arial', 'size': 12}
user_prefs = {'color': 'red', 'size': 14}
combined = ChainMap(user_prefs, defaults)   # user_prefs takes priority
print(f"color: {combined['color']}")  # 'red' — from user_prefs
print(f"font:  {combined['font']}")   # 'Arial' — from defaults

In [ ]:
# ── itertools — building blocks for iteration ────────────────────────────
import itertools

# count — infinite counter
counter = itertools.count(10, 2)   # 10, 12, 14, ...
print(list(itertools.islice(counter, 6)))

# cycle — repeat a sequence forever
colors = itertools.cycle(['red','green','blue'])
print([next(colors) for _ in range(7)])

# repeat — repeat a value N times (or forever)
print(list(itertools.repeat('x', 4)))

# chain — concatenate iterables
print(list(itertools.chain([1,2], [3,4], [5,6])))

# zip_longest — zip with fill value
print(list(itertools.zip_longest('ABC','xy', fillvalue='-')))

# combinations and permutations
print(list(itertools.combinations('ABCD', 2)))
print(list(itertools.permutations('ABC', 2)))
print(list(itertools.combinations_with_replacement('AB', 2)))

# product — cartesian product
print(list(itertools.product('AB', [1,2])))

# groupby — group consecutive items by a key
data = sorted([('A',1),('B',2),('A',3),('B',4),('C',5)], key=lambda x: x[0])
for key, group in itertools.groupby(data, key=lambda x: x[0]):
    print(f"  {key}: {list(group)}")

In [ ]:
# ── Other essential standard library modules (quick overview) ─────────────
# These deserve their own notebooks but here's a taste:

# abc — Abstract Base Classes
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self) -> float: ...

# dataclasses — generate __init__, __repr__, __eq__ automatically
from dataclasses import dataclass, field

@dataclass
class Product:
    name: str
    price: float
    tags: list = field(default_factory=list)

p = Product("Widget", 9.99, ["sale", "new"])
print(p)

# enum — symbolic names for a set of values
from enum import Enum, auto

class Direction(Enum):
    NORTH = auto()
    SOUTH = auto()
    EAST  = auto()
    WEST  = auto()

d = Direction.NORTH
print(f"{d} — {d.name} — {d.value}")

# typing — type hints support
from typing import TypeVar, Generic

# contextlib — context manager utilities (see Functions notebook)
from contextlib import contextmanager, suppress

# io — in-memory file-like objects
import io
buf = io.StringIO()
buf.write("hello")
buf.seek(0)
print(f"StringIO: {buf.read()}")

# copy — shallow and deep copies
import copy
original = [[1,2],[3,4]]
shallow  = copy.copy(original)     # new list, same inner lists
deep     = copy.deepcopy(original) # fully independent copy
print(f"Deep copy independent: {deep is not original and deep[0] is not original[0]}")

---

## 9. Virtual Environments & `pip`

A **virtual environment** is an isolated Python installation for a single project. It prevents package version conflicts between projects.

In [ ]:
# ── Creating and using a virtual environment ─────────────────────────────
# (Run these commands in your terminal, NOT in Python)

commands = """
# Create a virtual environment
python -m venv .venv

# Activate it
source .venv/bin/activate      # macOS / Linux
.venv\\Scripts\\activate        # Windows PowerShell

# Your prompt changes to show (.venv)
# Now install packages — they go into .venv, not your system Python
pip install requests
pip install "flask>=3.0"
pip install numpy pandas matplotlib

# Save your dependencies
pip freeze > requirements.txt

# Install from requirements.txt (e.g., after cloning a project)
pip install -r requirements.txt

# Deactivate the virtual environment
deactivate
"""
print(commands)

In [ ]:
# ── pip commands reference ────────────────────────────────────────────────
pip_commands = """
pip install package              # install latest version
pip install package==1.2.3       # exact version
pip install "package>=1.0,<2.0" # version range
pip install -U package           # upgrade to latest
pip install -e .                 # install current dir as editable package

pip uninstall package            # remove a package
pip list                         # show installed packages
pip show package                 # show package details
pip check                        # verify dependency compatibility

pip search package               # search PyPI (may be disabled)
"""
print(pip_commands)

In [ ]:
# ── pyproject.toml — the modern way to define a package ──────────────────
pyproject = """
# pyproject.toml (PEP 518 / PEP 621)
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.backends.legacy:build"

[project]
name = "mypackage"
version = "1.0.0"
description = "My awesome package"
readme = "README.md"
requires-python = ">=3.10"
dependencies = [
    "requests>=2.28",
    "click>=8.0",
]

[project.optional-dependencies]
dev = ["pytest", "ruff", "mypy"]

[project.scripts]
my-tool = "mypackage.cli:main"
"""
print(pyproject)

---

## 10. Relative vs Absolute Imports

Inside a package, you can import siblings and parents with **relative imports** (using `.` notation) instead of spelling out the full package path.

In [ ]:
# Absolute imports — always work from anywhere, recommended in most cases
#   from geometry.shapes import Circle
#   from geometry.transforms import rotate

# Relative imports — only work inside a package
#   from .shapes import Circle         # sibling module (same package)
#   from ..utils import helper         # parent package's utils module
#   from . import transforms           # sibling package

# In geometry/__init__.py we used:
#   from .shapes import Circle, Rectangle, Triangle
# The leading . means "same package as this file"

# Let's verify what our __init__.py does
with open('geometry/__init__.py') as f:
    print(f.read())

In [ ]:
# When to use which:
guidance = """
ABSOLUTE IMPORTS — prefer these:
  from geometry.shapes import Circle
  + Always unambiguous
  + Work in scripts and interactive sessions
  + Recommended by PEP 8

RELATIVE IMPORTS — use inside packages:
  from .shapes import Circle
  + Shorter when package name is long
  + Remain valid if the package is renamed
  - Only valid inside a package (not scripts)
  - Can confuse readers unfamiliar with the package
"""
print(guidance)

---

## 11. Lazy Imports & Performance

In [ ]:
# Measuring import time
import time

def timed_import(name):
    import sys
    # Remove from cache to force a fresh import
    sys.modules.pop(name, None)
    start = time.perf_counter()
    __import__(name)
    elapsed = time.perf_counter() - start
    return elapsed * 1000

for mod in ['json', 'os', 're', 'math', 'datetime']:
    ms = timed_import(mod)
    print(f"  import {mod:<12} {ms:.2f}ms")

print()
print("Subsequent imports use the cache (sys.modules) — essentially free:")
start = time.perf_counter()
import json   # already in sys.modules — instant
print(f"  import json (cached): {(time.perf_counter()-start)*1e6:.1f}µs")

In [ ]:
# Lazy import pattern — defer slow imports until actually needed

class LazyModule:
    """Import a module only on first attribute access."""
    def __init__(self, name):
        self._name   = name
        self._module = None

    def __getattr__(self, attr):
        if self._module is None:
            import importlib
            self._module = importlib.import_module(self._name)
        return getattr(self._module, attr)

# json is not imported yet
lazy_json = LazyModule('json')

# Only imported when we actually use it
result = lazy_json.dumps({"a": 1})
print(result)

# Python 3.12+ has importlib.util.LazyLoader for this purpose
import importlib.util
print(f"LazyLoader available: {hasattr(importlib.util, 'LazyLoader')}")

In [ ]:
# TYPE_CHECKING guard — import only for type checkers, not at runtime
# Avoids circular imports and speeds up startup

from typing import TYPE_CHECKING

if TYPE_CHECKING:
    # These imports only happen when mypy / pyright is analysing the code
    # They are NOT executed at runtime
    from pathlib import Path
    from collections import OrderedDict

def process_file(path: 'Path') -> None:   # use string annotation to avoid NameError
    """Process a file at the given path."""
    from pathlib import Path as _Path     # real import inside the function
    print(f"Processing: {_Path(path).name}")

process_file('geometry/shapes.py')

---

## 12. Useful Patterns — `importlib`, Namespace Packages

In [ ]:
# importlib — programmatic import utilities
import importlib

# Import a module by name string (useful for plugins)
mod_name = 'json'
json_mod = importlib.import_module(mod_name)
print(json_mod.dumps({"loaded": "dynamically"}))

# Import a specific attribute from a module by name
def import_from(module_name, attr_name):
    mod  = importlib.import_module(module_name)
    return getattr(mod, attr_name)

sqrt = import_from('math', 'sqrt')
print(f"sqrt(49) = {sqrt(49)}")

# importlib.metadata — inspect installed package metadata
import importlib.metadata
try:
    version = importlib.metadata.version('pip')
    print(f"pip version: {version}")
except importlib.metadata.PackageNotFoundError:
    print("pip metadata not found")

In [ ]:
# Plugin pattern — load modules dynamically at runtime
import os, importlib

# Create some fake plugin files
os.makedirs('plugins', exist_ok=True)

for name, code in [
    ('upper_plugin', 'def transform(s): return s.upper()'),
    ('reverse_plugin', 'def transform(s): return s[::-1]'),
    ('shout_plugin', 'def transform(s): return s + "!!!".upper()'),
]:
    with open(f'plugins/{name}.py', 'w') as f:
        f.write(code)

# Load all plugins from the directory
import sys
sys.path.insert(0, 'plugins')

plugins = {}
for fname in os.listdir('plugins'):
    if fname.endswith('.py') and not fname.startswith('_'):
        mod_name = fname[:-3]   # strip .py
        mod = importlib.import_module(mod_name)
        plugins[mod_name] = mod
        print(f"Loaded plugin: {mod_name}")

print()
text = "hello world"
for name, plugin in plugins.items():
    print(f"  {name}: {plugin.transform(text)}")

In [ ]:
# importlib.resources — access data files bundled with a package
import importlib.resources

# In a real package, you'd do:
# with importlib.resources.open_text('mypackage', 'data.json') as f:
#     data = json.load(f)

# This is the right way to bundle and read data files —
# NOT by constructing __file__-based paths
print("importlib.resources API:")
print(dir(importlib.resources))

In [ ]:
# Namespace packages (PEP 420) — packages without __init__.py
# Useful for splitting a large package across multiple directories / repos

explanation = """
Regular package:       Namespace package:

mypackage/             src1/mypackage/
├── __init__.py  ←     │   └── module_a.py
├── module_a.py        src2/mypackage/        (no __init__.py!)
└── module_b.py            └── module_b.py

With both src1 and src2 in sys.path, Python merges
them into a single 'mypackage' namespace.
Used by large frameworks to allow distributed plugins.
"""
print(explanation)

---

## 13. Quick Reference

| Topic | Key Syntax / Tool |
|---|---|
| Import a module | `import math` |
| Import with alias | `import numpy as np` |
| Import specific names | `from math import pi, sqrt` |
| Import with alias | `from datetime import datetime as DT` |
| All public names | `from module import *` (avoid in production) |
| Module search path | `sys.path` |
| Module cache | `sys.modules` |
| Reload a module | `importlib.reload(mod)` |
| Entry-point guard | `if __name__ == "__main__":` |
| List module contents | `dir(module)` |
| Public API | `__all__ = ["name1", "name2"]` |
| Dynamic import | `importlib.import_module("name")` |
| Lazy import | Import inside a function or class |
| Type-checking only | `if TYPE_CHECKING: import ...` |
| Virtual environment | `python -m venv .venv` |
| Install package | `pip install package` |
| Save dependencies | `pip freeze > requirements.txt` |
| Relative import | `from .sibling import name` |

In [ ]:
# 🏆 Practice: Build a mini plugin system with stdlib modules

import json, re, math
from datetime import datetime
from collections import Counter
from pathlib import Path

def analyse_text(text: str) -> dict:
    """
    Analyse a block of text using several stdlib modules.
    Returns a dict of statistics.
    """
    words = re.findall(r"\b[a-zA-Z']+\b", text.lower())
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    word_freq  = Counter(words)

    avg_word_len = sum(len(w) for w in words) / len(words) if words else 0

    return {
        "analysed_at":   datetime.now().strftime("%Y-%m-%d %H:%M"),
        "char_count":    len(text),
        "word_count":    len(words),
        "unique_words":  len(word_freq),
        "sentence_count":len(sentences),
        "avg_word_len":  round(avg_word_len, 2),
        "top_5_words":   word_freq.most_common(5),
        "lexical_diversity": round(len(word_freq) / len(words), 3) if words else 0,
    }

sample = """
Python is a versatile programming language. Python is known for its clear
syntax and readability. Many data scientists and web developers use Python
every day. Python is powerful yet beginner-friendly.
"""

result = analyse_text(sample)

print("=" * 40)
print("       TEXT ANALYSIS REPORT")
print("=" * 40)
for key, value in result.items():
    print(f"  {key:<22}: {value}")

# Save the result as JSON
Path('analysis.json').write_text(json.dumps(result, indent=2))
print("\nReport saved to analysis.json")